# 3D MRT Permeability Notebook (Configurable Geometry + Pressure BC)

This notebook runs single-phase **3D MRT LBM** permeability estimation and lets you choose:
- geometry (`spheres` or `cylinders`)
- pressure boundary condition (`zouhe` or `regularized`)


In [ ]:
import os
import numpy as np

from examples.singlephase.permeability_mrt_zouhe import (
    LatticeD3Q19,
    PorousPermeabilityMRT3D,
    build_cylinder_pack_mask,
    build_sphere_pack_mask,
    d3q19_mrt_matrix,
)


In [ ]:
# User controls
geometry = 'spheres'      # 'spheres' or 'cylinders'
pressure_bc = 'zouhe'     # 'zouhe' or 'regularized'
precision = 'f32/f32'
nx, ny, nz = 120, 72, 72
omega = 1.3
rho0 = 1.0
delta_rho = 1e-3
nsteps = 10000
io_rate = 1000


In [ ]:
if geometry == 'spheres':
    solid_mask = build_sphere_pack_mask(nx, ny, nz)
elif geometry == 'cylinders':
    solid_mask = build_cylinder_pack_mask(nx, ny, nz)
else:
    raise ValueError("geometry must be 'spheres' or 'cylinders'")

porosity = 1.0 - np.mean(solid_mask)
print(f'Geometry={geometry}, pressure_bc={pressure_bc}, porosity={porosity:.4f}')


In [ ]:
os.system('rm -rf output* *.vtk')

kwargs = {
    'lattice': LatticeD3Q19(precision),
    'omega': omega,
    'nx': nx,
    'ny': ny,
    'nz': nz,
    'M': d3q19_mrt_matrix(),
    's_rho': 0.0,
    's_e': 1.1,
    's_eta': 1.0,
    's_j': 0.0,
    's_q': 1.0,
    's_v': omega,
    's_m': 1.0,
    's_pi': 1.0,
    'precision': precision,
    'io_rate': io_rate,
    'print_info_rate': io_rate,
    'checkpoint_rate': -1,
    'checkpoint_dir': os.path.abspath('./checkpoints'),
    'restore_checkpoint': False,
    'solid_mask': solid_mask,
    'rho0': rho0,
    'delta_rho': delta_rho,
    'pressure_bc': pressure_bc,
}

sim = PorousPermeabilityMRT3D(**kwargs)
sim.run(nsteps)
